
## Univariada: experiencia laboral profesional (WorkExp) vs. 12 años

**Hipótesis:** los desarrolladores argentinos tienen, en promedio, más de 12 años de experiencia laboral profesional.

**Variable:** WorkExp (cuantitativa, años).

**Hipótesis:**
- **H0:** μ = 12 años
- **H1:** μ > 12 años

**Justificación:** 12 años permite distinguir perfiles con recorrido consolidado de perfiles junior o intermedios. Además, la media global de `WorkExp` ronda los 12,7 años, por lo que el umbral es una referencia razonable.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scipy.stats import shapiro

df_clean = pd.read_pickle("df_clean.pkl")

# 1. Filtrar por Argentina
df_arg = df_clean[df_clean["Country"] == "Argentina"].copy()


# 3. Configurar y crear el gráfico
plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")

# Graficar la distribución de los años de experiencia
sns.histplot(data=df_arg, x="WorkExp", bins=20, color="blue", edgecolor="black")

# Configurar títulos y etiquetas
plt.title("Distribución de Años de Experiencia (WorkExp) en Argentina", fontsize=14, pad=15)
plt.xlabel("Años de Experiencia Profesional", fontsize=12)
plt.ylabel("Cantidad de Desarrolladores", fontsize=12)

# Mostrar la leyenda para identificar las líneas
plt.legend(fontsize=11)

# Ajustar el diseño y mostrar
plt.tight_layout()
plt.show()

#Validacion de hipotesis

# Grafico qqplot para analizar la normalidad de forma visual
workexp = df_arg["WorkExp"].dropna()

plt.figure(figsize=(8, 5))
stats.probplot(workexp, dist="norm", plot=plt)

plt.title("Gráfico Q-Q de WorkExp (Argentina)", fontsize=14)
plt.xlabel("Cuantiles Teóricos (Distribución Normal)", fontsize=12)
plt.ylabel("Valores Reales de WorkExp", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

#Analizamos normalidad con el test de Shapiro-wilk
estadistico, p_valor = stats.shapiro(workexp)

print("--- Test de Shapiro-Wilk ---")
print(f"Estadístico: {estadistico}")
print(f"Valor p: {p_valor}")

if p_valor > 0.05:
    print("Conclusión: No se rechaza la normalidad")
else:
    print("Conclusión: Se rechaza la normalidad (Los datos NO siguen una distribución normal).")

Tanto el el grafico como el test de shapiro-wikl muestran que la variable workExp no sigue normalidad. De todas formas sabemos que los t-test son considerablemente robustos cuando el tamaño de las muestras es igual o mayor a 30. Por lo que igual podemos aplicar el test.

In [ ]:
# 2. Definir los parámetros del test
valor_hipotesis = 12


# 3. Ejecutar el T-test de una muestra
# alternative='greater' indica que probamos si la media poblacional es MAYOR a 12
t_stat, p_value = stats.ttest_1samp(workexp, popmean=valor_hipotesis, alternative='greater')

# 4. Mostrar los resultados numéricos
print(f"Media muestral: {workexp.mean():.2f} años")
print(f"Estadístico T: {t_stat:.4f}")
print(f"Valor p (p-value): {p_value:.4e}\n")

# 5. Interpretar el resultado estadístico
if p_value < 0.05:
    print("Conclusión: Rechazamos la hipótesis nula.")
    print("Hay evidencia estadística suficiente para afirmar que los desarrolladores argentinos tienen, en promedio, más de 12 años de experiencia.")
else:
    print("Conclusión: No podemos rechazar la hipótesis nula.")
    print("No hay evidencia estadística suficiente para afirmar que la media supera los 12 años.")
    

# Bivariada: El impacto de las herramientas de IA en la compensación salarial.
    
Población de estudio: Desarrolladores.

Hipótesis Nula (H0): La compensación salarial mediana de los desarrolladores que utilizan IA es igual o menor a la de quienes no la utilizan.

Hipótesis Alternativa (H1): La compensación salarial mediana de los desarrolladores que utilizan IA es estrictamente mayor a la de quienes no la utilizan.
    
Atributos involucrados: AIDevHaveWorkedWith (herramientas de IA utilizadas), ConvertedCompYearly (salario anual estandarizado en USD).

La industria tecnológica asume que los asistentes de IA aumentan drásticamente la productividad ("10x developer effect" mediante IA). Desde el punto de vista de los datos, probar esta hipótesis permite determinar si el mercado laboral ya está recompensando económicamente esta eficiencia extra, o si, por el contrario, la estandarización del código generado por IA está comoditizando el desarrollo y aplanando los salarios


In [ ]:
# Me quedo con los desarrolladores independientes
empleo = ["Independent contractor, freelancer, or self-employed"]
df_emp = df_clean[df_clean["Employment"].isin(empleo)].copy()

condiciones = [df_emp["AISelect"].str.startswith("Yes", na=False),df_emp["AISelect"].str.startswith("No", na=False),]
opciones = ["Usa IA", "No usa IA"]

df_emp["uso_ia"] = np.select(condiciones, opciones, default="Sin respuesta")

salarioAnual = "ConvertedCompYearly"

#Elimino los nan de los salrios
df_emp = df_emp.dropna(subset=[salarioAnual]).copy()

# Filtrar registros válidos excluyendo 'Sin respuesta'
df_plot = df_emp[
    (df_emp["uso_ia"].isin(["Usa IA", "No usa IA"]))].copy()

#Gráfico
plt.figure(figsize=(9, 6))

# Boxplot base para ver la dispersión y la mediana
sns.boxplot(
    data=df_plot,
    x="uso_ia",
    y=salarioAnual,
    palette=["#B0BEC5", "#1E88E5"],
    width=0.35,
    showfliers=False,
)

plt.title(
    f"Compensación Salarial según Uso de IA)",
    fontsize=12,
    fontweight="bold",
    pad=15,
)
plt.xlabel("Uso Activo de IA", fontsize=10)
plt.ylabel("Compensación Salarial Anual (USD)", fontsize=10)
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

En el grafico podemos que no hay gran diferencia los grupos.


In [ ]:
grupo_ia = df_emp[df_emp["uso_ia"] == "Usa IA"][salarioAnual]
grupo_no_ia = df_emp[df_emp["uso_ia"] == "No usa IA"][salarioAnual]

# SUPUESTO 1: NORMALIDAD

# A. Análisis Gráfico: Q-Q Plot
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

stats.probplot(grupo_ia, dist="norm", plot=ax[0])
ax[0].set_title("Q-Q Plot: Desarrolladores que Usan IA")

stats.probplot(grupo_no_ia, dist="norm", plot=ax[1])
ax[1].set_title("Q-Q Plot: Desarrolladores que NO Usan IA")

plt.tight_layout()
plt.show()

Vemos que los datos no siguen una distribucion normal. Por lo que realizaremos el test de mannwhitney U

In [ ]:


stat, p_valor = stats.mannwhitneyu(x=grupo_ia, y=grupo_no_ia, alternative='greater')

# 3. Mostrar los resultados
print("--- Test de Mann-Whitney U ---")
print(f"Estadístico U: {stat}")
print(f"p-valor: {p_valor:.4e}\n")

if p_valor < 0.05:
    print("Conclusión: Rechazamos la Hipótesis Nula (H0).")
    print("Existe evidencia estadística significativa para afirmar que los desarrolladores que utilizan herramientas de IA tienen una compensación salarial mayor.")
else:
    print("Conclusión: No podemos rechazar la Hipótesis Nula (H0).")
    print("No existe evidencia estadística suficiente para afirmar que el uso de herramientas de IA impacte positivamente en el salario.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import scipy.stats as stats

# 1. Carga de datos limpios
df_clean = pd.read_pickle("df_clean.pkl")
cols_multivariada = ["YearsCode", "WorkExp", "ToolCountWork", "ToolCountPersonal"]

# 2. Filtrado de nulos y estandarización
df_cluster = df_clean[cols_multivariada].dropna().copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster[cols_multivariada])

# 3. Evaluación de clusters (k = 2 a 6)
k_range = range(2, 7)
inercias, silhouettes = [], []

for k in k_range:
    kmeans_eval = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_eval = kmeans_eval.fit_predict(X_scaled)
    inercias.append(kmeans_eval.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels_eval))

# 4. Gráficos de diagnóstico (Codo y Silueta)
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
sns.set_style("whitegrid")

ax[0].plot(k_range, inercias, marker='o', color='#1E88E5', linewidth=2)
ax[0].set_title("Método del Codo (Inercia)", fontsize=13, fontweight='bold')
ax[0].set_xlabel("Número de Clusters (k)", fontsize=11)
ax[0].set_ylabel("Inercia Intra-cluster", fontsize=11)

ax[1].plot(k_range, silhouettes, marker='s', color='#D81B60', linewidth=2)
ax[1].set_title("Coeficiente de Silueta según k", fontsize=13, fontweight='bold')
ax[1].set_xlabel("Número de Clusters (k)", fontsize=11)
ax[1].set_ylabel("Silhouette Score", fontsize=11)

plt.tight_layout()
plt.show()

# Hipotesis 1:
Los desarrolladores que respondieron la encuesta de Argentina tienen
un salario 3 veces mayor al Salario Mínimo, Vital y Móvil (SMVM)

Variable: `ConvertedCompYearly` (salario anual estandarizado en USD).

Actualmente, el SMVM es de $398.800 mensuales. Al calcular el salario anual (considerando el aguinaldo) y multiplicarlo por 3, obtenemos un total de $15.553.200. Tomando una cotización del dólar actual aproximada de $1.600, esto equivale a USD 9.720,75 anuales.


In [ ]:
# Configurar el estilo visual (opcional pero recomendado)
sns.set_theme(style="whitegrid")

# ---------------------------------------------------------
# Opción 1: Histograma (Distribución de los salarios)
# ---------------------------------------------------------
plt.figure(figsize=(10, 6))
sns.histplot(data=df_arg, x='ConvertedCompYearly', bins=30, color='blue')

plt.title('Distribución de Salarios Anuales en Argentina (USD)', fontsize=14)
plt.xlabel('Salario Anual (USD)', fontsize=12)
plt.ylabel('Frecuencia', fontsize=12)

plt.show()


In [ ]:
salarios_arg = df_arg['ConvertedCompYearly'].dropna()

valor_hipotesis = 9720.75

#  alternative='greater' para evaluar si la media poblacional es MAYOR a 9720.75
t_stat, p_value = stats.ttest_1samp(a=salarios_arg, popmean=valor_hipotesis, alternative='greater')

print(f"Estadístico T: {t_stat:.4f}")
print(f"Valor P (p-value): {p_value:.4e}\n")

if p_value < 0.05:
    print("Conclusión: Rechazamos la hipótesis nula (H0).")
    print("Hay evidencia estadística suficiente para afirmar que el salario promedio de los desarrolladores en Argentina es significativamente mayor a 3 veces el SMVM (USD 9720.75).")
else:
    print("Conclusión: No podemos rechazar la hipótesis nula (H0).")
    print("No hay evidencia estadística para afirmar que el salario promedio es superior a 3 veces el SMVM (USD 9720.75).")

# Hipotesis 2
Los años de experiencia profesional influyen en la confianza en la
precisión de las herramientas de IA

In [ ]:
categorias_confianza = df_clean['AIAcc'].dropna().unique().tolist()
print(categorias_confianza)

plt.figure(figsize=(10, 6))

# Crear el boxplot
sns.boxplot(
    data=df_clean,
    x='AIAcc',
    y='WorkExp',
    order=categorias_confianza,
    palette='Blues',
    showfliers=False  # Oculta la columna de puntos outlayers
)

plt.title('Distribución de Años de Experiencia según el Nivel de Confianza en la IA', fontsize=14)
plt.xlabel('Nivel de Confianza en las Herramientas de IA (AIAcc)', fontsize=12)
plt.ylabel('Años de Experiencia Profesional (YearsCodePro)', fontsize=12)
plt.xticks(rotation=15)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()